In [ ]:
# ============================================================
# Portable Research Platform Bootstrap - Canonical v1.2
# ============================================================
"""
MANDATORY FIRST CELL.

One notebook version works in:
- local VS Code / Jupyter;
- Google Drive desktop sync;
- Google Colab with Drive mounted;
- Colab transient clone under /content.

Best practice: keep the full research_platform_definitive folder on Google Drive at
MyDrive/machine-learning-for-trading/research_platform_definitive or
MyDrive/GitHub/machine-learning-for-trading/research_platform_definitive.
If Colab cannot find it, this cell can clone the GitHub repo into /content as a fallback.
"""

from pathlib import Path
import os
import subprocess
import sys


DEFAULT_GIT_URL = os.environ.get(
    "RESEARCH_PLATFORM_GIT_URL",
    "https://github.com/TheGenesisAIStory/ml-trading-thesis-bot.git",
)


def _has_platform_sentinel(path):
    path = Path(path).expanduser()
    return (
        (path / "src" / "research_platform_core").exists()
        or (path / "src" / "research_platform_core.py").exists()
    )


def _candidate_roots():
    cwd = Path.cwd().resolve()
    candidates = []

    env_root = os.environ.get("RESEARCH_PLATFORM_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    drive_desktop_candidates = [
        Path.home() / "Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/GitHub/machine-learning-for-trading/research_platform_definitive",
        Path.home() / "Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/machine-learning-for-trading/research_platform_definitive",
    ]

    local_mirror_candidates = []
    for p in [cwd, *cwd.parents]:
        local_mirror_candidates.append(p)
        local_mirror_candidates.append(p / "research_platform_definitive")
    local_mirror_candidates.append(Path.home() / "GitHub/machine-learning-for-trading/research_platform_definitive")

    colab_candidates = [
        Path("/content/drive/MyDrive/GitHub/machine-learning-for-trading/research_platform_definitive"),
        Path("/content/drive/MyDrive/machine-learning-for-trading/research_platform_definitive"),
        Path("/content/drive/MyDrive/research_platform_definitive"),
        Path("/content/machine-learning-for-trading/research_platform_definitive"),
        Path("/content/ml-trading-thesis-bot/research_platform_definitive"),
        Path("/content/research_platform_definitive"),
    ]

    prefer_drive = os.environ.get("RESEARCH_PLATFORM_STORAGE_MODE", "drive").strip().lower() != "local"
    if _is_colab():
        candidates.extend(colab_candidates)
        candidates.extend(drive_desktop_candidates)
        candidates.extend(local_mirror_candidates)
    elif prefer_drive:
        candidates.extend(drive_desktop_candidates)
        candidates.extend(local_mirror_candidates)
        candidates.extend(colab_candidates)
    else:
        candidates.extend(local_mirror_candidates)
        candidates.extend(drive_desktop_candidates)
        candidates.extend(colab_candidates)

    deduped = []
    seen = set()
    for p in candidates:
        key = str(p.expanduser())
        if key not in seen:
            deduped.append(p)
            seen.add(key)
    return deduped


def _find_project_root():
    for candidate in _candidate_roots():
        candidate = candidate.expanduser()
        if _has_platform_sentinel(candidate):
            return candidate.resolve()
        nested = candidate / "research_platform_definitive"
        if _has_platform_sentinel(nested):
            return nested.resolve()
    return None


def _is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _mount_drive_if_colab(verbose=True):
    if not _is_colab():
        return
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            if verbose:
                print("Mounting Google Drive...")
            drive.mount("/content/drive")
    except Exception as exc:
        if verbose:
            print(f"Drive mount skipped/failed: {exc}")


def _clone_repo_fallback(verbose=True):
    if not _is_colab():
        return None
    if os.environ.get("RESEARCH_PLATFORM_AUTO_CLONE", "1") in {"0", "false", "False"}:
        return None

    target = Path(os.environ.get("RESEARCH_PLATFORM_CLONE_ROOT", "/content/machine-learning-for-trading"))
    if _has_platform_sentinel(target / "research_platform_definitive"):
        return (target / "research_platform_definitive").resolve()

    if target.exists() and not (target / ".git").exists():
        return None

    try:
        if target.exists():
            if verbose:
                print(f"Updating existing clone: {target}")
            subprocess.run(["git", "-C", str(target), "pull", "--ff-only"], check=False)
        else:
            if verbose:
                print(f"Cloning research platform repo into {target}...")
            subprocess.run(["git", "clone", "--depth", "1", DEFAULT_GIT_URL, str(target)], check=True)
    except Exception as exc:
        if verbose:
            print(f"Git clone fallback failed: {exc}")
        return None

    root = target / "research_platform_definitive"
    return root.resolve() if _has_platform_sentinel(root) else None


def _first_existing_path(candidates, default):
    for candidate in candidates:
        candidate = Path(candidate).expanduser()
        if candidate.exists():
            return candidate
    return default


def _ensure_writable_dir(path, fallback):
    for candidate in [Path(path).expanduser(), Path(fallback).expanduser(), Path("/tmp/research_platform_output")]:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            probe = candidate / ".write_test"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink(missing_ok=True)
            return candidate
        except Exception:
            continue
    raise OSError("No writable output/cache directory available.")


def setup_colab_environment(verbose=True):
    _mount_drive_if_colab(verbose=verbose)
    project_root = _find_project_root()
    if project_root is None:
        project_root = _clone_repo_fallback(verbose=verbose)

    if project_root is None:
        searched = "\n".join(f"- {p.expanduser()}" for p in _candidate_roots())
        raise FileNotFoundError(
            "PROJECT_ROOT not found. This notebook needs the full research_platform_definitive folder, not only the notebook.\n\n"
            "Best fix: sync this folder to Google Drive:\n"
            "  MyDrive/machine-learning-for-trading/research_platform_definitive\n\n"
            "Alternative: set RESEARCH_PLATFORM_GIT_URL and let Colab clone the repo into /content.\n\n"
            f"Searched:\n{searched}"
        )

    for rel in ["", "src", "company_valuation/src", "portfolio_analysis/src"]:
        path = str(project_root / rel)
        if path not in sys.path:
            sys.path.insert(0, path)

    financial_db_root = _first_existing_path(
        [
            Path(os.environ.get("FINANCIAL_DB_ROOT", "")) if os.environ.get("FINANCIAL_DB_ROOT") else Path("__missing__"),
            Path("/content/drive/MyDrive/Database Finanziario"),
            Path.home() / "Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/Database Finanziario",
        ],
        Path("/content/drive/MyDrive/Database Finanziario") if _is_colab() else project_root / "local_databases_not_on_drive" / "database",
    )

    output_root = _ensure_writable_dir(
        Path(os.environ.get("RESEARCH_PLATFORM_OUTPUT_ROOT", project_root / "output")),
        Path("/content/research_platform_output") if _is_colab() else project_root / "output",
    )
    local_cache = _ensure_writable_dir(
        Path(os.environ.get("RESEARCH_PLATFORM_LOCAL_CACHE", output_root / "data_cache")),
        Path("/content/research_platform_cache") if _is_colab() else output_root / "data_cache",
    )

    config = {
        "environment": "colab" if _is_colab() else "local",
        "PROJECT_ROOT": project_root,
        "FINANCIAL_DB_ROOT": financial_db_root,
        "DB_BASE": financial_db_root,
        "DATA_PATH": financial_db_root,
        "OUTPUTROOT": output_root,
        "OUTPUT_ROOT": output_root,
        "LOCAL_CACHE_ROOT": local_cache,
        "DATA_LOCAL": local_cache,
    }

    for key in ["FINANCIAL_DB_ROOT", "DB_BASE", "DATA_PATH", "RESEARCH_PLATFORM_OUTPUT_ROOT", "RESEARCH_PLATFORM_LOCAL_CACHE", "DATA_LOCAL", "COMPANY_VALUATION_DATA_LOCAL"]:
        if key in {"RESEARCH_PLATFORM_OUTPUT_ROOT"}:
            os.environ[key] = str(output_root)
        elif key in {"RESEARCH_PLATFORM_LOCAL_CACHE", "DATA_LOCAL", "COMPANY_VALUATION_DATA_LOCAL"}:
            os.environ[key] = str(local_cache)
        else:
            os.environ[key] = str(financial_db_root)

    globals().update(config)

    if verbose:
        print(f"PROJECT_ROOT: {project_root}")
        print(f"Environment: {config['environment']}")
        print(f"FINANCIAL_DB_ROOT: {financial_db_root} | exists={financial_db_root.exists()}")
        print(f"OUTPUTROOT: {output_root}")
        print(f"LOCAL_CACHE_ROOT: {local_cache}")
        print("sys.path project entries inserted: OK")
    return config


CONFIG = setup_colab_environment(verbose=True)

try:
    from research_platform_core import read_dataset, resolve_dataset_path
    from ml_stock_lab import features, valuation
    from smart_money_engine import run_smart_money_engine
    print("Core imports: OK")
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Core imports failed after bootstrap: {exc}. Confirm PROJECT_ROOT contains src/research_platform_core and src/ml_stock_lab."
    ) from exc


# 0. ML Stock Lab Experiments

Colab-first, notebook-first laboratory for ML equity valuation, model-implied mispricing, factor overlays, AQR factor panels and quintile diagnostics.

Use this notebook when you want a guided research workflow. Use the Streamlit app when you want to review artifacts and run orchestration jobs.


In [ ]:
# 0.1 Colab / Local Bootstrap - mandatory first code cell
from pathlib import Path
import os
import sys


def setup_colab_environment(verbose: bool = True) -> dict:
    """Resolve project paths for Colab, VS Code and local notebook execution."""
    config: dict = {}
    try:
        from google.colab import drive  # type: ignore
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        config['environment'] = 'colab'
    except Exception:
        config['environment'] = 'local'

    candidates = [
        Path('/content/drive/MyDrive/GitHub/machine-learning-for-trading/research_platform_definitive'),
        Path('/content/drive/MyDrive/GitHub/machine-learning-for-trading'),
        Path('/content/drive/MyDrive/machine-learning-for-trading/research_platform_definitive'),
        Path('/content/machine-learning-for-trading/research_platform_definitive'),
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    project_root = None
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if (candidate / 'src' / 'ml_stock_lab' / '__init__.py').exists() and (candidate / 'src' / 'research_platform_core').exists():
            project_root = candidate
            break
        nested = candidate / 'research_platform_definitive'
        if (nested / 'src' / 'ml_stock_lab' / '__init__.py').exists() and (nested / 'src' / 'research_platform_core').exists():
            project_root = nested.resolve()
            break
    if project_root is None:
        raise FileNotFoundError(
            'PROJECT_ROOT not found. In Colab, clone or sync the full repository, not only this notebook.'
        )

    for rel in ['', 'src', 'company_valuation/src', 'portfolio_analysis/src']:
        path = str(project_root / rel)
        if path not in sys.path:
            sys.path.insert(0, path)

    financial_db = Path(os.environ.get('FINANCIAL_DB_ROOT', '/content/drive/MyDrive/Database Finanziario'))
    if config['environment'] == 'local':
        local_drive = Path.home() / 'Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/Database Finanziario'
        if local_drive.exists():
            financial_db = local_drive

    output_root = project_root / 'output' / 'ml_stock_lab'
    config.update({
        'PROJECT_ROOT': project_root,
        'FINANCIAL_DB_ROOT': financial_db,
        'OUTPUTROOT': output_root,
        'TABLESDIR': output_root / 'tables',
        'FIGURESDIR': output_root / 'figures',
    })
    config['TABLESDIR'].mkdir(parents=True, exist_ok=True)
    config['FIGURESDIR'].mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('FINANCIAL_DB_ROOT', str(financial_db))
    globals().update(config)

    if verbose:
        print('Environment:', config['environment'])
        print('PROJECT_ROOT:', project_root)
        print('FINANCIAL_DB_ROOT:', financial_db)
        print('OUTPUTROOT:', output_root)
    return config


CONFIG = setup_colab_environment(verbose=True)


## 1. Imports

This notebook imports the canonical package-backed APIs. If imports fail here, the repository path is not correctly mounted.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML, clear_output

try:
    from src.ml_stock_lab import (
        FundamentalDatasetBuilder,
        ExpectedReturnModel,
        PeerImpliedValuator,
        add_basic_features,
        compute_absolute_mispricing,
        compute_relative_mispricing,
        cross_sectional_zscore,
        describe_temporal_split,
        evaluate_quintile_backtest,
        load_aqr_factor_panel,
        load_financial_db_panel,
        make_forward_returns,
        make_quantile_portfolios,
        oos_r2,
        rank_scores,
        run_ml_stock_lab_experiment,
        select_numeric_features,
        temporal_train_test_split,
        top_bottom,
        validate_panel_coverage,
    )
    from src.research_platform_core import AqrFactorProvider, get_aqr_factor_panel
except ModuleNotFoundError:
    from ml_stock_lab import (
        FundamentalDatasetBuilder,
        ExpectedReturnModel,
        PeerImpliedValuator,
        add_basic_features,
        compute_absolute_mispricing,
        compute_relative_mispricing,
        cross_sectional_zscore,
        describe_temporal_split,
        evaluate_quintile_backtest,
        load_aqr_factor_panel,
        load_financial_db_panel,
        make_forward_returns,
        make_quantile_portfolios,
        oos_r2,
        rank_scores,
        run_ml_stock_lab_experiment,
        select_numeric_features,
        temporal_train_test_split,
        top_bottom,
        validate_panel_coverage,
    )
    from research_platform_core import AqrFactorProvider, get_aqr_factor_panel

print('ML Stock Lab imports OK')


## 2. Fintech Control Center

Choose universe, model family, optional AQR factors, API keys and experiment limits. Press **Apply** before running the cells below.


In [ ]:
# 2.1 User-friendly ML Lab controls
try:
    import ipywidgets as widgets
    WIDGETS_OK = True
except Exception:
    widgets = None
    WIDGETS_OK = False

API_KEY_FIELDS = {
    'FMP_API_KEY': 'Financial Modeling Prep',
    'FINNHUB_API_KEY': 'Finnhub',
    'ALPHA_VANTAGE_API_KEY': 'Alpha Vantage',
    'EODHD_API_KEY': 'EODHD',
    'FRED_API_KEY': 'FRED',
}

ML_LAB_UNIVERSES = {
    'All Available': [],
    'US Mega Cap': ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'LLY', 'JPM'],
    'Semiconductors': ['NVDA', 'AMD', 'AVGO', 'QCOM', 'INTC', 'TSM', 'ASML'],
    'Europe Quality': ['ASML.AS', 'SAP.DE', 'RMS.PA', 'MC.PA', 'NOVO-B.CO', 'NESN.SW'],
    'Italy Banks': ['ISP.MI', 'UCG.MI', 'BAMI.MI', 'BMED.MI', 'MB.MI'],
    'Manual': [],
}

PROFILE_PRESETS = {
    'Balanced': {'model': 'ols', 'feature_blocks': ['value', 'quality', 'momentum'], 'target': 'market_value'},
    'Value': {'model': 'lasso', 'feature_blocks': ['value', 'quality'], 'target': 'market_value'},
    'Quality Growth': {'model': 'rf', 'feature_blocks': ['quality', 'growth', 'momentum'], 'target': 'market_value'},
    'Factor Research': {'model': 'ensemble', 'feature_blocks': ['value', 'quality', 'momentum', 'risk', 'aqr'], 'target': 'market_value'},
}

DEFAULT_CONFIG = {
    'profile': 'Balanced',
    'universe': 'All Available',
    'tickers': [],
    'model': 'ols',
    'target': 'market_value',
    'feature_blocks': ['value', 'quality', 'momentum'],
    'max_rows': 2000,
    'min_tickers': 5,
    'min_dates': 2,
    'include_aqr': False,
    'aqr_slugs': ['quality_minus_junk_factors_monthly'],
    'aqr_max_datasets': 1,
    'refresh_cache': False,
}
ML_LAB_CONFIG = {**DEFAULT_CONFIG, **globals().get('ML_LAB_CONFIG', {})}


def parse_tickers(text: str) -> list[str]:
    out = []
    for token in str(text or '').replace('\n', ',').replace(';', ',').split(','):
        ticker = token.strip().upper()
        if ticker and ticker not in out:
            out.append(ticker)
    return out


def parse_lines(text: str) -> list[str]:
    return [x.strip() for x in str(text or '').replace('\n', ',').split(',') if x.strip()]


def apply_env_keys(text: str) -> list[str]:
    applied = []
    for line in str(text or '').splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        sep = '=' if '=' in line else ':' if ':' in line else None
        if not sep:
            continue
        key, value = line.split(sep, 1)
        key = key.strip().upper()
        value = value.strip().strip('"').strip("'")
        if key in API_KEY_FIELDS and value:
            os.environ[key] = value
            applied.append(key)
    return applied


def api_key_status_html() -> str:
    rows = []
    for key, label in API_KEY_FIELDS.items():
        configured = bool(os.environ.get(key))
        badge = 'configured' if configured else 'missing'
        color = '#067647' if configured else '#b54708'
        bg = '#ecfdf3' if configured else '#fffaeb'
        rows.append(f"<tr><td>{label}</td><td><code>{key}</code></td><td style='background:{bg};color:{color};font-weight:700'>{badge}</td></tr>")
    return """
    <div style='border:1px solid #d0d5dd;border-radius:8px;padding:12px;background:#fff'>
    <b>API key status</b><table style='width:100%;border-collapse:collapse;margin-top:8px'>
    <tr style='background:#01696f;color:white'><th>Provider</th><th>Env var</th><th>Status</th></tr>
    """ + ''.join(rows) + '</table></div>'


def sync_config(cfg: dict) -> dict:
    global ML_LAB_CONFIG, MODEL, MAX_ROWS, MIN_TICKERS, MIN_DATES
    ML_LAB_CONFIG = cfg
    MODEL = str(cfg['model'])
    MAX_ROWS = int(cfg['max_rows'])
    MIN_TICKERS = int(cfg['min_tickers'])
    MIN_DATES = int(cfg['min_dates'])
    globals().update({'MODEL': MODEL, 'MAX_ROWS': MAX_ROWS, 'MIN_TICKERS': MIN_TICKERS, 'MIN_DATES': MIN_DATES})
    return cfg

sync_config(ML_LAB_CONFIG)

if not WIDGETS_OK:
    display(HTML("<div style='background:#fff7ed;border-left:5px solid #da7101;padding:12px;border-radius:8px'>ipywidgets unavailable. Edit <code>ML_LAB_CONFIG</code> manually and continue.</div>"))
    display(HTML(api_key_status_html()))
else:
    style = {'description_width': '130px'}
    profile_w = widgets.Dropdown(options=list(PROFILE_PRESETS), value=ML_LAB_CONFIG['profile'], description='Profile', style=style, layout=widgets.Layout(width='300px'))
    universe_w = widgets.Dropdown(options=list(ML_LAB_UNIVERSES), value=ML_LAB_CONFIG['universe'], description='Universe', style=style, layout=widgets.Layout(width='300px'))
    tickers_w = widgets.Textarea(value=', '.join(ML_LAB_CONFIG.get('tickers', [])), description='Tickers', placeholder='AAPL, MSFT, NVDA or leave empty for all available artifacts', style=style, layout=widgets.Layout(width='760px', height='80px'))
    model_w = widgets.Dropdown(options=['ols', 'lasso', 'rf', 'gbrt', 'ensemble'], value=ML_LAB_CONFIG['model'], description='Model', style=style, layout=widgets.Layout(width='260px'))
    target_w = widgets.Dropdown(options=['market_value', 'forward_return'], value=ML_LAB_CONFIG['target'], description='Target', style=style, layout=widgets.Layout(width='260px'))
    features_w = widgets.SelectMultiple(options=['value', 'quality', 'growth', 'momentum', 'risk', 'aqr', 'model_based'], value=tuple(ML_LAB_CONFIG['feature_blocks']), description='Feature blocks', style=style, layout=widgets.Layout(width='460px', height='130px'))
    max_rows_w = widgets.IntSlider(value=int(ML_LAB_CONFIG['max_rows']), min=100, max=20000, step=100, description='Max rows', style=style, layout=widgets.Layout(width='460px'))
    include_aqr_w = widgets.Checkbox(value=bool(ML_LAB_CONFIG['include_aqr']), description='Include AQR factor panel', indent=False)
    aqr_slugs_w = widgets.Textarea(value=', '.join(ML_LAB_CONFIG['aqr_slugs']), description='AQR slugs', style=style, layout=widgets.Layout(width='760px', height='70px'))
    aqr_max_w = widgets.IntSlider(value=int(ML_LAB_CONFIG['aqr_max_datasets']), min=1, max=20, step=1, description='AQR max', style=style, layout=widgets.Layout(width='460px'))
    refresh_w = widgets.Checkbox(value=bool(ML_LAB_CONFIG['refresh_cache']), description='Refresh cached data where safe', indent=False)
    keys_w = widgets.Textarea(value='', placeholder='Paste keys as KEY=value, one per line. Example:\nFMP_API_KEY=...\nFRED_API_KEY=...', description='API keys', style=style, layout=widgets.Layout(width='760px', height='95px'))
    apply_w = widgets.Button(description='Apply configuration', icon='check', button_style='success', layout=widgets.Layout(width='210px', height='38px'))
    status_out = widgets.Output()

    def on_profile(change=None):
        preset = PROFILE_PRESETS.get(profile_w.value, {})
        model_w.value = preset.get('model', model_w.value)
        target_w.value = preset.get('target', target_w.value)
        features_w.value = tuple(preset.get('feature_blocks', list(features_w.value)))
        if 'aqr' in preset.get('feature_blocks', []):
            include_aqr_w.value = True

    def on_universe(change=None):
        values = ML_LAB_UNIVERSES.get(universe_w.value, [])
        if values:
            tickers_w.value = ', '.join(values)

    def on_apply(_=None):
        with status_out:
            clear_output(wait=True)
            cfg = {
                'profile': profile_w.value,
                'universe': universe_w.value,
                'tickers': parse_tickers(tickers_w.value),
                'model': model_w.value,
                'target': target_w.value,
                'feature_blocks': list(features_w.value),
                'max_rows': int(max_rows_w.value),
                'min_tickers': int(ML_LAB_CONFIG.get('min_tickers', 5)),
                'min_dates': int(ML_LAB_CONFIG.get('min_dates', 2)),
                'include_aqr': bool(include_aqr_w.value),
                'aqr_slugs': parse_lines(aqr_slugs_w.value),
                'aqr_max_datasets': int(aqr_max_w.value),
                'refresh_cache': bool(refresh_w.value),
            }
            applied = apply_env_keys(keys_w.value)
            keys_w.value = ''
            sync_config(cfg)
            print('Configuration applied')
            print(cfg)
            print('API keys updated:', applied if applied else 'none')
            display(HTML(api_key_status_html()))

    profile_w.observe(on_profile, names='value')
    universe_w.observe(on_universe, names='value')
    apply_w.on_click(on_apply)

    display(HTML("""
    <div style='background:#f6fbfb;border:1px solid #d0d5dd;border-left:6px solid #01696f;border-radius:10px;padding:14px;margin:8px 0'>
      <h3 style='margin:0;color:#01696f'>ML Stock Lab Control Center</h3>
      <p style='margin:6px 0 0 0'>Choose the universe, model, feature blocks and optional AQR factors. Then run the one-click experiment cell.</p>
    </div>
    """))
    display(widgets.VBox([
        widgets.HBox([profile_w, universe_w, model_w, target_w]),
        tickers_w,
        widgets.HBox([features_w, max_rows_w]),
        widgets.HBox([include_aqr_w, aqr_max_w, refresh_w]),
        aqr_slugs_w,
        keys_w,
        apply_w,
        status_out,
    ]))
    display(HTML(api_key_status_html()))


## 3. One-Click Experiment Runner

This is the easiest path. It runs the canonical package function, writes all `MLStockLab_*` artifacts, and returns DataFrames for interactive inspection.


In [ ]:
# 3.1 Run the canonical ML Stock Lab artifact pipeline
run_result = run_ml_stock_lab_experiment(
    output_root=OUTPUTROOT,
    financial_db_root=FINANCIAL_DB_ROOT,
    model=ML_LAB_CONFIG['model'],
    max_rows=int(ML_LAB_CONFIG['max_rows']),
    target=ML_LAB_CONFIG['target'],
    feature_blocks=ML_LAB_CONFIG['feature_blocks'],
    universe=ML_LAB_CONFIG['universe'],
    tickers=ML_LAB_CONFIG['tickers'] or None,
    min_tickers=int(ML_LAB_CONFIG['min_tickers']),
    min_dates=int(ML_LAB_CONFIG['min_dates']),
    include_aqr=bool(ML_LAB_CONFIG['include_aqr']),
    aqr_slugs=ML_LAB_CONFIG['aqr_slugs'] or None,
    aqr_max_datasets=int(ML_LAB_CONFIG['aqr_max_datasets']) if ML_LAB_CONFIG.get('aqr_max_datasets') else None,
)
print('Run status:', run_result.get('status'))
print('Artifact paths:')
for key, path in (run_result.get('paths') or {}).items():
    print(f'- {key}: {path}')
panel = run_result.get('panel', pd.DataFrame())
signals = run_result.get('signals', pd.DataFrame())
metrics = run_result.get('metrics', pd.DataFrame())
quintile_returns = run_result.get('quintiles', pd.DataFrame())
prediction_metrics = run_result.get('prediction_metrics', pd.DataFrame())
display(metrics)


## 4. Data Coverage and AQR Factor Preview

Review what the model actually saw. Missing data is a research finding, not something to hide.


In [ ]:
coverage = validate_panel_coverage(panel, min_tickers=ML_LAB_CONFIG['min_tickers'], min_dates=ML_LAB_CONFIG['min_dates']) if isinstance(panel, pd.DataFrame) else pd.DataFrame()
display(coverage)

if ML_LAB_CONFIG.get('include_aqr'):
    aqr_panel = load_aqr_factor_panel(
        slugs=ML_LAB_CONFIG.get('aqr_slugs') or None,
        max_datasets=int(ML_LAB_CONFIG.get('aqr_max_datasets') or 1),
        financial_db_root=FINANCIAL_DB_ROOT,
        output_root=PROJECT_ROOT / 'output',
    )
    print('AQR panel shape:', aqr_panel.shape)
    display(aqr_panel.tail())
else:
    aqr_panel = pd.DataFrame()
    print('AQR disabled. Enable it in the Control Center for factor overlays.')

display(panel.head(10) if isinstance(panel, pd.DataFrame) and not panel.empty else pd.DataFrame())


## 5. Feature and Signal Diagnostics

These tables show which numeric features were selected and how the fair-value/mispricing signal ranks securities.


In [ ]:
if panel.empty:
    feature_cols = []
    print('No panel available.')
else:
    feature_cols = select_numeric_features(panel, target='market_value', min_non_null=max(3, min(10, len(panel)//10)))
    if not feature_cols:
        feature_cols = [c for c in panel.select_dtypes('number').columns if c not in {'market_value', 'forward_return'}][:12]
    print('Selected feature count:', len(feature_cols))
    display(pd.DataFrame({'feature': feature_cols}))

if signals.empty:
    print('No signals available yet.')
else:
    display(signals.sort_values('zscore' if 'zscore' in signals.columns else signals.columns[-1], ascending=False).head(25))


## 6. Model Comparison

Quick model-family comparison using the same feature matrix. This is diagnostic, not a production model selection routine.


In [ ]:
comparison_rows = []
if panel.empty or not feature_cols:
    comparison_df = pd.DataFrame([{'model': ML_LAB_CONFIG['model'], 'status': 'skipped', 'reason': 'empty panel or features'}])
else:
    X, y, used_features = FundamentalDatasetBuilder(feature_cols, 'market_value').build(panel)
    for model_name in ['ols', 'lasso', 'rf', 'gbrt', 'ensemble']:
        try:
            pred = PeerImpliedValuator(model=model_name).fit_predict(X, y)
            mis = compute_relative_mispricing(pred, y)
            comparison_rows.append({
                'model': model_name,
                'status': 'ok',
                'rows': len(X),
                'features': len(used_features),
                'fair_value_corr': pred.corr(y),
                'mispricing_std': mis.std(),
            })
        except Exception as exc:
            comparison_rows.append({'model': model_name, 'status': 'failed', 'error': f'{type(exc).__name__}: {exc}'})
    comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(TABLESDIR / 'MLStockLab_model_comparison.csv', index=False)
display(comparison_df)


## 7. Visual Dashboard

Interactive Plotly charts for the core research outputs.


In [ ]:
if signals.empty:
    print('No signals to plot.')
else:
    if {'market_value', 'fair_value_hat'}.issubset(signals.columns):
        fig = px.scatter(
            signals,
            x='market_value',
            y='fair_value_hat',
            color='zscore' if 'zscore' in signals.columns else None,
            hover_name='ticker' if 'ticker' in signals.columns else None,
            title='ML Fair Value vs Market Value',
            template='plotly_white',
            color_continuous_scale='RdYlGn',
        )
        max_axis = pd.to_numeric(signals[['market_value', 'fair_value_hat']].stack(), errors='coerce').max()
        if pd.notna(max_axis):
            fig.add_trace(go.Scatter(x=[0, max_axis], y=[0, max_axis], mode='lines', name='Fair = Market', line=dict(color='#667085', dash='dash')))
        fig.write_html(FIGURESDIR / 'MLStockLab_fair_value_vs_market.html')
        fig.show()

    if 'zscore' in signals.columns:
        fig2 = px.histogram(signals, x='zscore', nbins=35, title='Mispricing z-score distribution', template='plotly_white', color_discrete_sequence=['#01696f'])
        fig2.add_vline(x=0, line_dash='dash', line_color='#667085')
        fig2.write_html(FIGURESDIR / 'MLStockLab_zscore_distribution.html')
        fig2.show()

    if {'ticker', 'zscore'}.issubset(signals.columns):
        top_bottom_frame = pd.concat([
            signals.sort_values('zscore', ascending=False).head(15).assign(bucket='Top'),
            signals.sort_values('zscore', ascending=True).head(15).assign(bucket='Bottom'),
        ])
        fig3 = px.bar(top_bottom_frame, x='ticker', y='zscore', color='bucket', title='Top / Bottom by ML mispricing z-score', template='plotly_white', color_discrete_map={'Top': '#01696f', 'Bottom': '#da7101'})
        fig3.write_html(FIGURESDIR / 'MLStockLab_top_bottom_scores.html')
        fig3.show()

if isinstance(quintile_returns, pd.DataFrame) and not quintile_returns.empty and {'quantile', 'return'}.issubset(quintile_returns.columns):
    fig4 = px.bar(quintile_returns, x='quantile', y='return', color='quantile', title='Quintile / Long-short return diagnostic', template='plotly_white')
    fig4.write_html(FIGURESDIR / 'MLStockLab_quintile_returns.html')
    fig4.show()


## 8. Quintile and Prediction Diagnostics

Quintile tests are research diagnostics. For production use, add transaction costs, liquidity limits and independent factor attribution.


In [ ]:
if isinstance(quintile_returns, pd.DataFrame) and not quintile_returns.empty:
    qmetrics = evaluate_quintile_backtest(quintile_returns)
else:
    qmetrics = pd.DataFrame()
qmetrics.to_csv(TABLESDIR / 'MLStockLab_quintile_metrics.csv', index=False)
display(qmetrics)

display(prediction_metrics if isinstance(prediction_metrics, pd.DataFrame) else pd.DataFrame())


## 9. Export Browser

Everything below `OUTPUTROOT` is consumed by Streamlit and orchestration.


In [ ]:
exports = sorted([p for p in OUTPUTROOT.rglob('*') if p.is_file()])
export_df = pd.DataFrame({
    'relative_path': [str(p.relative_to(OUTPUTROOT)) for p in exports],
    'size_kb': [round(p.stat().st_size / 1024, 2) for p in exports],
    'modified': [pd.Timestamp(p.stat().st_mtime, unit='s').isoformat() for p in exports],
})
display(export_df)


## 10. Interpretation Notes

- `fair_value_hat` is an ML-implied cross-sectional estimate, not a DCF replacement.
- `mispricing_rel` compares model fair value to observed market value.
- `zscore` is the cross-sectional standardized signal used for ranking and quintiles.
- AQR factors are date-level overlays; they support factor context and attribution, not single-name fundamentals.
- Missing data and sparse quintiles should stop promotion to production.


In [ ]:
summary = {
    'status': run_result.get('status'),
    'project_root': str(PROJECT_ROOT),
    'financial_db_root': str(FINANCIAL_DB_ROOT),
    'output_root': str(OUTPUTROOT),
    'config': ML_LAB_CONFIG,
    'panel_rows': len(panel) if isinstance(panel, pd.DataFrame) else 0,
    'signal_rows': len(signals) if isinstance(signals, pd.DataFrame) else 0,
    'feature_count': len(feature_cols) if 'feature_cols' in globals() else 0,
}
summary_path = OUTPUTROOT / 'MLStockLab_experiment_summary.json'
import json
summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
summary


## 11. Next Experiments

Suggested next runs:

1. Run `Balanced` without AQR for a simple baseline.
2. Run `Factor Research` with QMJ + BAB AQR factors.
3. Compare OLS/LASSO against RF/GBRT/ensemble.
4. Promote only signals that survive coverage, quintile, risk and factor diagnostics.
